# Phase 5 — Output & Evaluation

**Primary model: REDUCED feature set (slope, tri removed)**, per the decision after Phase 4 —
the reduced set is the more honest signal given the negative-sampling terrain confound documented
there. The full feature set is not re-analyzed here; if you want its SHAP for comparison (to show how
much weight slope/tri absorb), that's a quick copy of Section 2 with `full_feature_cols`.

Sections: (1) headline metrics recap, (2) SHAP interpretability, (3) ROC/PR/confusion diagnostics,
(4) susceptibility raster + uncertainty layer (runs in Colab with Earth Engine).

## 0. Setup and re-fit final model

**Note on why this re-fits rather than loading `phase4_final_models.joblib`:** joblib-pickled sklearn
objects are tied to the sklearn version that saved them — loading `phase4_final_models.joblib` in a
different sklearn version raises `AttributeError` on internal attributes (confirmed: Colab saved it
under sklearn 1.6.1, this failed to unpickle under 1.8.0). Rather than pin exact versions across
sessions, this re-fits the exact same final model (same hyperparameters Phase 4 already selected) from
`phase3_feature_table.csv`, which has no such fragility. Takes a few seconds, not the 20+ minute
per-point GEE extraction — no Earth Engine calls happen here.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              roc_curve, precision_recall_curve, confusion_matrix)
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

df = pd.read_csv("phase3_feature_table.csv")
oof = pd.read_csv("phase4_oof_predictions.csv")

REDUCED_FEATURE_COLS = ['aspect', 'curvature', 'elevation', 'twi', 'BSI', 'NDVI', 'NDWI',
                         'spectral_window_days', 'VH_post', 'VV_post', 'dVH', 'dVV',
                         'API', 'rain_1d', 'rain_30d', 'rain_7d']

X = df[REDUCED_FEATURE_COLS]
y = df["label"]

rf_final = Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("clf", RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42))])
rf_final.fit(X, y)

scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
xgb_final = xgb.XGBClassifier(eval_metric="logloss", scale_pos_weight=scale_pos_weight, random_state=42,
                               tree_method="hist", n_estimators=500, max_depth=5, learning_rate=0.05)
xgb_final.fit(X, y)

print("Refit complete. Feature count:", len(REDUCED_FEATURE_COLS))


## 1. Headline metrics (recap from Phase 4, reduced set = primary)

These are the **out-of-fold** metrics from Phase 4's Spatial Cluster K-Fold — not in-sample scores from
the models just refit above (those would be optimistic; the refit models above are for SHAP/raster use,
not for reporting performance). Repeated here as the primary Phase 5 headline numbers, with the same
terrain-confound caveat from Phase 4 still in force: treat as an upper bound, not validated field skill.

In [ ]:
y_oof = oof["label"]
p_rf, p_xgb = oof["oof_rf_reduced"], oof["oof_xgb_reduced"]

print("=== REDUCED feature set -- primary Phase 5 result ===")
print(f"RF   -- AUROC: {roc_auc_score(y_oof, p_rf):.4f}  PR-AUC: {average_precision_score(y_oof, p_rf):.4f}  F1@0.5: {f1_score(y_oof, p_rf>0.5):.4f}")
print(f"XGB  -- AUROC: {roc_auc_score(y_oof, p_xgb):.4f}  PR-AUC: {average_precision_score(y_oof, p_xgb):.4f}  F1@0.5: {f1_score(y_oof, p_xgb>0.5):.4f}")
print()
print("Reminder: both numbers are inflated by the negative-sampling terrain confound (Phase 4 caveat).")
print("Read as 'can distinguish landslide-prone terrain from artificially-flat stable terrain', not yet")
print("validated as 'given comparably unstable terrain, ranks susceptibility correctly'.")


## 2. SHAP interpretability (reduced model)

Using `TreeExplainer` on the refit RF and XGB (reduced feature set). RF's `shap_values` returns one
array per class for binary classification in recent SHAP versions -- we take the positive-class (label=1)
slice.

In [ ]:
X_imputed = pd.DataFrame(rf_final.named_steps["impute"].transform(X), columns=REDUCED_FEATURE_COLS)

explainer_rf = shap.TreeExplainer(rf_final.named_steps["clf"])
shap_values_rf = explainer_rf.shap_values(X_imputed)
shap_values_rf_pos = shap_values_rf[:, :, 1] if np.array(shap_values_rf).ndim == 3 else shap_values_rf

explainer_xgb = shap.TreeExplainer(xgb_final)
shap_values_xgb = explainer_xgb.shap_values(X)

print("Random Forest SHAP summary (reduced feature set):")
shap.summary_plot(shap_values_rf_pos, X_imputed, show=True)


In [ ]:
print("XGBoost SHAP summary (reduced feature set):")
shap.summary_plot(shap_values_xgb, X, show=True)


**Reading this against the Phase 4 caveat:** if `elevation`, `VH_post`, `VV_post`, `twi`, or `curvature`
dominate the SHAP ranking, treat that with the same suspicion as slope/tri in the full set — those are
exactly the features flagged as partial terrain-confound proxies. If `API`/`rain_*`/`NDVI`/`NDWI` rank
highly instead, that's closer to the genuine rainfall-susceptibility signal the project is actually
after.

## 3. ROC / PR curves + confusion matrix (out-of-fold, reduced set)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for name, p in [("RF", p_rf), ("XGB", p_xgb)]:
    fpr, tpr, _ = roc_curve(y_oof, p)
    axes[0].plot(fpr, tpr, label=f"{name} (AUROC={roc_auc_score(y_oof, p):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve (OOF)"); axes[0].legend()

for name, p in [("RF", p_rf), ("XGB", p_xgb)]:
    prec, rec, _ = precision_recall_curve(y_oof, p)
    axes[1].plot(rec, prec, label=f"{name} (PR-AUC={average_precision_score(y_oof, p):.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curve (OOF)"); axes[1].legend()

cm = confusion_matrix(y_oof, p_xgb > 0.5)
axes[2].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
axes[2].set_xticks([0, 1]); axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(["Pred 0", "Pred 1"]); axes[2].set_yticklabels(["True 0", "True 1"])
axes[2].set_title("Confusion matrix (XGB, thresh=0.5, OOF)")

plt.tight_layout()
plt.savefig("phase5_diagnostic_plots.png", dpi=150)
plt.show()
print("Saved phase5_diagnostic_plots.png")


## 4. Susceptibility raster + uncertainty layer

**This section requires Earth Engine and runs in Colab, not here.** It follows the same pattern as
Phase 3's static topographic extraction: one batched call per source over a regular grid of points
across the ROI, not per-pixel loops.

**Uncertainty layer:** two complementary measures, both cheap to compute from what's already fit:
- **RF tree disagreement** -- standard deviation of individual tree probabilities within the Random
  Forest. High std = the trees disagree = the model itself is uncertain at that location.
- **RF/XGB disagreement** -- absolute difference between the two models' probabilities. High
  disagreement = the result is sensitive to model choice, which is its own kind of "don't trust this
  number too much" signal, independent of RF's internal variance.

Neither is a calibrated confidence interval -- they're both practical proxies standard in susceptibility
mapping when a bare probability map would otherwise look more certain than the small (254-point),
spatially clustered training set actually supports.

In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='landslidedetection-506008')  # <-- your GEE-linked Cloud project

lat_min, lat_max = 26.0, 35.0
lon_min, lon_max = 74.0, 97.0
roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

# Coarser grid than the 254 training points -- e.g. 0.1 degree (~11km) spacing keeps this to a
# manageable number of GEE round trips. Tighten GRID_STEP_DEG later for a higher-res production map
# once this runs cleanly at low resolution.
GRID_STEP_DEG = 0.5  # coarse first pass -- ~55km cells, ~3hr at observed rate.
# Drop back to 0.1 later for production resolution ONLY after switching dynamic
# extraction to Export.table.toDrive() batch export -- per-point getInfo() calls
# measured ~13s/point in testing, making the full 20,700-point grid a ~75hr job.
grid_lats = np.arange(lat_min, lat_max, GRID_STEP_DEG)
grid_lons = np.arange(lon_min, lon_max, GRID_STEP_DEG)
grid_points = [(lon, lat) for lat in grid_lats for lon in grid_lons]
print(f"Grid: {len(grid_points)} points at {GRID_STEP_DEG} deg spacing")


In [ ]:
# --- Static topographic bands (CHUNKED -- a single sampleRegions().getInfo() call over this many
# points times out; Phase 3's 254-point single call worked because it was ~80x smaller) ---
srtm = ee.Image("USGS/SRTMGL1_003").select("elevation").clip(roi)
srtm_aspect = ee.Terrain.aspect(srtm)  # BUGFIX: this was accidentally dropped earlier --
# only slope/tri were meant to be excluded from the reduced feature set, aspect stays.
curvature = srtm.convolve(ee.Kernel.laplacian8(normalize=False)).rename("curvature")
merit_hydro = ee.Image("MERIT/Hydro/v1_0_1")
upstream_area_m2 = merit_hydro.select("upa").multiply(1e6)
slope_rad = ee.Terrain.slope(srtm).multiply(np.pi / 180)
twi = upstream_area_m2.divide(slope_rad.tan().max(1e-6)).log().rename("twi")

topo_image = (srtm.rename("elevation")
              .addBands(srtm_aspect.rename("aspect"))
              .addBands(curvature)
              .addBands(twi))  # slope/tri intentionally excluded -- reduced feature set

CHUNK_SIZE = 1000  # points per sampleRegions() call -- tune down further if this still times out
MAX_RETRIES = 4
BASE_BACKOFF_SECONDS = 2

def call_with_retry(fn, *args, **kwargs):
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            return fn(*args, **kwargs), None
        except Exception as e:
            last_error = str(e)[:300]
            time.sleep(BASE_BACKOFF_SECONDS * (2 ** attempt))
    return None, last_error

import time
topo_chunks = []
n_chunks = (len(grid_points) + CHUNK_SIZE - 1) // CHUNK_SIZE
for c in range(n_chunks):
    chunk = grid_points[c * CHUNK_SIZE : (c + 1) * CHUNK_SIZE]
    points_fc = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([lon, lat]), {"idx": c * CHUNK_SIZE + i})
        for i, (lon, lat) in enumerate(chunk)
    ])

    def _sample():
        fc = topo_image.sampleRegions(collection=points_fc, scale=90, geometries=False, tileScale=4)
        return fc.getInfo()["features"]

    features, error = call_with_retry(_sample)
    if error:
        print(f"Chunk {c+1}/{n_chunks} FAILED after {MAX_RETRIES} retries: {error}")
        continue
    topo_chunks.extend(features)
    print(f"Chunk {c+1}/{n_chunks} done ({len(features)} points)")

topo_records = [f["properties"] for f in topo_chunks]
topo_grid_df = pd.DataFrame(topo_records).set_index("idx").sort_index()
print(f"\nTopographic grid extraction: {len(topo_grid_df)} / {len(grid_points)} points")


In [ ]:
# --- Present-day dynamic bands: single latest composite per grid point (not per historical event) ---
# Functions below are copied from Phase3_Feature_Engineering.ipynb (Section 2) with ONE necessary
# change: the original SAR pre/post windowing is built around a historical event_date (average
# backscatter in the 12 days BEFORE the event vs the 12 days AFTER it). For a live susceptibility
# snapshot at END_DATE = today, a literal "post" window would need data from the future, which
# doesn't exist. Redefined here as two consecutive recent windows ending at END_DATE instead:
#   "post" = most recent SAR_WINDOW_DAYS up to today (current conditions)
#   "pre"  = the SAR_WINDOW_DAYS window immediately before that (recent baseline)
# dVV/dVH then represent recent short-term backscatter change rather than change around a specific
# triggering event -- the closest meaningful analogue available without future data. Spectral and
# rainfall functions need no such change since they already only look backward from END_DATE.

END_DATE = ee.Date(pd.Timestamp.today().strftime("%Y-%m-%d"))
SAR_WINDOW_DAYS = 12
SPECTRAL_WINDOW_STEPS = [30, 60, 90, 120]
MAX_RETRIES = 4
BASE_BACKOFF_SECONDS = 2

def call_with_retry(fn, *args, **kwargs):
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            return fn(*args, **kwargs), None
        except Exception as e:
            last_error = str(e)[:300]
            time.sleep(BASE_BACKOFF_SECONDS * (2 ** attempt))
    return None, last_error


def mask_s2_clouds_and_shadows(image, max_cloud_prob=20):
    """Masks clouds (s2cloudless) and cloud shadows (SCL + NIR/SWIR dark-pixel check).
    Identical to Phase 3 -- no change needed."""
    cloud_prob = ee.Image(image.get('s2cloudless')).select('probability')
    is_cloud = cloud_prob.gte(max_cloud_prob)
    scl = image.select('SCL')
    is_shadow_scl = scl.eq(3)
    dark_pixels = image.select('B8').lt(1500).And(image.select('B11').lt(1000))
    shadow_near_cloud = is_cloud.focal_max(radius=60, units='meters').And(dark_pixels)
    combined_mask = is_cloud.Or(is_shadow_scl).Or(shadow_near_cloud)
    optical_bands = image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']).divide(10000)
    return optical_bands.updateMask(combined_mask.Not()).copyProperties(image, ['system:time_start'])


# Same S2/S1 collection definitions as Phase 3, filtered to this ROI (not per-event date range,
# since a live map needs whatever imagery is most recent, not a fixed historical S2_START/S2_END).
s2_sr_grid = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi)
s2_cloud_prob_grid = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(roi)
s2_join_grid = ee.Join.saveFirst('s2cloudless').apply(
    primary=s2_sr_grid, secondary=s2_cloud_prob_grid,
    condition=ee.Filter.equals(leftField='system:index', rightField='system:index'))
s2_with_prob_grid = ee.ImageCollection(s2_join_grid)

s1_grd_grid = (ee.ImageCollection("COPERNICUS/S1_GRD")
               .filterBounds(roi)
               .filter(ee.Filter.eq("instrumentMode", "IW"))
               .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
               .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
               .select(["VV", "VH"]))

chirps_grid = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(roi)


def get_spectral_features_grid(lon, lat, end):
    """Identical logic to Phase 3's get_spectral_features -- only the source of `end` changes
    (a shared END_DATE instead of each row's own event_date)."""
    point = ee.Geometry.Point([lon, lat])
    for window in SPECTRAL_WINDOW_STEPS:
        start = end.advance(-window, "day")
        window_coll = s2_with_prob_grid.filterDate(start, end)
        masked = window_coll.map(mask_s2_clouds_and_shadows)
        composite = masked.median()
        ndvi = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
        ndwi = composite.normalizedDifference(["B3", "B8"]).rename("NDWI")
        bsi = composite.expression(
            "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
            {"SWIR": composite.select("B11"), "RED": composite.select("B4"),
             "NIR": composite.select("B8"), "BLUE": composite.select("B2")}
        ).rename("BSI")
        img = ndvi.addBands(ndwi).addBands(bsi)
        result, err = call_with_retry(
            lambda: img.reduceRegion(reducer=ee.Reducer.first(), geometry=point, scale=10, bestEffort=True).getInfo()
        )
        if err is not None:
            raise RuntimeError(f"window={window}d: {err}")
        if result.get("NDVI") is not None:
            result["spectral_window_days"] = window
            return result
    return {"spectral_window_days": None}


def get_sar_features_grid(lon, lat, end):
    """Adapted: two consecutive recent windows ending at `end`, not a pre/post-event split --
    see note above on why the original pre/post definition can't be used for a live map."""
    point = ee.Geometry.Point([lon, lat])
    recent_post = s1_grd_grid.filterDate(end.advance(-SAR_WINDOW_DAYS, "day"), end).mean()
    recent_pre = s1_grd_grid.filterDate(end.advance(-2 * SAR_WINDOW_DAYS, "day"),
                                         end.advance(-SAR_WINDOW_DAYS, "day")).mean()
    delta = recent_post.subtract(recent_pre)
    img = (recent_pre.rename(["VV_pre", "VH_pre"])
           .addBands(recent_post.rename(["VV_post", "VH_post"]))
           .addBands(delta.rename(["dVV", "dVH"])))
    result, err = call_with_retry(
        lambda: img.reduceRegion(reducer=ee.Reducer.first(), geometry=point, scale=10, bestEffort=True).getInfo()
    )
    if err is not None:
        raise RuntimeError(err)
    return result


# BUGFIX: CHIRPS/DAILY ("final" product) has real publication latency -- typically several weeks --
# so filtering up to literal "today" (END_DATE) returns EMPTY collections for the 1-day and 7-day
# windows specifically. reduceRegion() then silently drops those bands from its result rather than
# erroring, which is why rain_1d/rain_7d came back entirely missing (not just NaN) across every point,
# while rain_30d/API still worked -- their windows are wide enough to catch older, already-published
# days. Fix: anchor rainfall to CHIRPS's own most recent actually-available date, found once and
# reused for every point, rather than the shared END_DATE used for spectral/SAR (which don't have
# this latency problem).
_chirps_latest_date = chirps_grid.sort("system:time_start", False).first().date()
print("CHIRPS latest available date:", _chirps_latest_date.format("YYYY-MM-dd").getInfo())

def get_rainfall_features_grid(lon, lat, end=None, api_decay=0.9):
    """Identical logic to Phase 3's get_rainfall_features, but anchored to CHIRPS's own latest
    available date (see bugfix note above) instead of the shared END_DATE."""
    point = ee.Geometry.Point([lon, lat])
    rain_end = _chirps_latest_date
    r1 = chirps_grid.filterDate(rain_end.advance(-1, "day"), rain_end).sum().rename("rain_1d")
    r7 = chirps_grid.filterDate(rain_end.advance(-7, "day"), rain_end).sum().rename("rain_7d")
    r30 = chirps_grid.filterDate(rain_end.advance(-30, "day"), rain_end).sum().rename("rain_30d")
    api_window = chirps_grid.filterDate(rain_end.advance(-30, "day"), rain_end)

    def weight(img):
        days_before = rain_end.difference(img.date(), "day")
        return img.multiply(ee.Number(api_decay).pow(days_before))

    api = api_window.map(weight).sum().rename("API")
    img = r1.addBands(r7).addBands(r30).addBands(api)
    result, err = call_with_retry(
        lambda: img.reduceRegion(reducer=ee.Reducer.first(), geometry=point, scale=5566, bestEffort=True).getInfo()
    )
    if err is not None:
        raise RuntimeError(err)
    return result


# --- Runtime reality check before running this for real ---
# Phase 3 took ~1337s for 254 points x 3 sources -- about 5.3s/point. At the ~19,785 points that
# survived topo extraction, that's roughly 29 HOURS run sequentially, which will outlast any single
# Colab session at full resolution. CHECKPOINT_PATH below saves progress every CHECKPOINT_EVERY
# points and the loop skips already-completed indices on re-run, so a disconnect mid-run doesn't
# lose progress. GRID_STEP_DEG is set to 0.5 above for this first pass (~828 points, ~3hr) --
# the checkpoint filename is tagged with the resolution so it won't collide with a later run
# at a different GRID_STEP_DEG.

CHECKPOINT_PATH = f"phase5_dynamic_grid_checkpoint_{GRID_STEP_DEG}deg.csv"  # resolution-
# tagged so an old checkpoint from a different GRID_STEP_DEG can never be silently reused --
# grid_points index i means a different (lon, lat) at each resolution.
CHECKPOINT_EVERY = 200

def extract_all_dynamic_grid(points, end_date, checkpoint_path=CHECKPOINT_PATH, checkpoint_every=CHECKPOINT_EVERY):
    try:
        done_df = pd.read_csv(checkpoint_path).set_index("idx")
        done_idx = set(done_df.index)
        print(f"Resuming from checkpoint: {len(done_idx)} points already done.")
        records = done_df.reset_index().to_dict("records")
    except FileNotFoundError:
        done_idx = set()
        records = []
        print("No checkpoint found -- starting fresh.")

    n = len(points)
    start_time = time.time()
    for i, (lon, lat) in enumerate(points):
        if i in done_idx:
            continue
        rec = {"idx": i, "longitude": lon, "latitude": lat}
        try:
            rec.update(get_spectral_features_grid(lon, lat, end_date))
        except Exception as e:
            rec["_spectral_error"] = str(e)[:200]
        try:
            rec.update(get_sar_features_grid(lon, lat, end_date))
        except Exception as e:
            rec["_sar_error"] = str(e)[:200]
        try:
            rec.update(get_rainfall_features_grid(lon, lat, end_date))
        except Exception as e:
            rec["_rainfall_error"] = str(e)[:200]
        records.append(rec)

        if (i + 1) % checkpoint_every == 0 or (i + 1) == n:
            pd.DataFrame(records).set_index("idx").to_csv(checkpoint_path)
            elapsed = time.time() - start_time
            print(f"  {len(records)}/{n} points done ({elapsed:.0f}s elapsed this session) -- checkpoint saved")

    return pd.DataFrame(records).set_index("idx")


dynamic_grid_df = extract_all_dynamic_grid(grid_points, END_DATE)
error_cols = [c for c in dynamic_grid_df.columns if c.startswith("_")]
if error_cols:
    print(f"\nPoints with a recorded extraction error, by source:")
    print(dynamic_grid_df[error_cols].notna().sum())
print(f"\nDynamic grid extraction: {len(dynamic_grid_df)} / {len(grid_points)} points")


### Patch: fix rainfall only, reusing the existing checkpoint

Your spectral + SAR extraction already succeeded for 828/828 points (the ~1.5hr run) -- only rainfall
had the CHIRPS-latency bug above. This cell re-runs **just** `get_rainfall_features_grid` against the
points already in `phase5_dynamic_grid_checkpoint_0.5deg.csv`, overwrites the (currently-missing)
`rain_1d`/`rain_7d`/`rain_30d`/`API` columns, and leaves every other already-extracted column untouched.
Much faster than a full re-run -- one source instead of three.

In [ ]:
# Re-run rainfall only against the existing checkpoint (spectral + SAR untouched)
checkpoint_df = pd.read_csv(CHECKPOINT_PATH).set_index("idx")
print(f"Loaded checkpoint: {len(checkpoint_df)} points")

rainfall_cols = ["rain_1d", "rain_7d", "rain_30d", "API"]
already_has_rainfall = all(c in checkpoint_df.columns for c in rainfall_cols) and \
                       checkpoint_df[rainfall_cols].notna().any().all() if \
                       all(c in checkpoint_df.columns for c in rainfall_cols) else False
if already_has_rainfall:
    print("Rainfall columns already present and populated -- nothing to patch.")
else:
    n = len(checkpoint_df)
    start_time = time.time()
    for count, (idx, row) in enumerate(checkpoint_df.iterrows()):
        lon, lat = row["longitude"], row["latitude"]
        try:
            rain_result = get_rainfall_features_grid(lon, lat)
            for k, v in rain_result.items():
                checkpoint_df.loc[idx, k] = v
            if "_rainfall_error" in checkpoint_df.columns:
                checkpoint_df.loc[idx, "_rainfall_error"] = np.nan
        except Exception as e:
            checkpoint_df.loc[idx, "_rainfall_error"] = str(e)[:200]

        if (count + 1) % 200 == 0 or (count + 1) == n:
            checkpoint_df.to_csv(CHECKPOINT_PATH)
            elapsed = time.time() - start_time
            print(f"  {count + 1}/{n} rainfall points patched ({elapsed:.0f}s elapsed) -- checkpoint saved")

    checkpoint_df.to_csv(CHECKPOINT_PATH)
    print(f"\nRainfall patch complete. Re-run 'dynamic_grid_df = ...' cell above -- it will load this")
    print("patched checkpoint (all points already done, so it returns immediately, no new EE calls).")


**Note on the SAR adaptation above:** the `get_sar_features_grid` pre/post windowing differs from Phase 3's `get_sar_features` for a real reason (explained in the code comments) -- a live map has no future post-event data to pull from, so it uses two consecutive recent windows ending at `END_DATE` instead. `get_spectral_features_grid` and `get_rainfall_features_grid` are otherwise identical in logic to their Phase 3 counterparts, just parameterized on `END_DATE` instead of each row's own `event_date`.

In [ ]:
# Once dynamic_grid_df exists (from the cell above), join to the topo grid and predict.
# inner join since a point needs BOTH topo and dynamic data to be usable -- this needs no further
# EE calls, so it's safe to re-run repeatedly while iterating on the model side.

grid_df = topo_grid_df.join(dynamic_grid_df, how="inner")
print(f"Points with both topo + dynamic data: {len(grid_df)} / {len(grid_points)}")

missing_feature_pct = grid_df[REDUCED_FEATURE_COLS].isna().mean().sort_values(ascending=False) * 100
print("\nMissingness per feature on the grid:")
print(missing_feature_pct.round(1))

grid_X = grid_df[REDUCED_FEATURE_COLS]
grid_X_imputed = pd.DataFrame(rf_final.named_steps["impute"].transform(grid_X),
                               columns=REDUCED_FEATURE_COLS, index=grid_df.index)

susceptibility_rf = rf_final.predict_proba(grid_X)[:, 1]
susceptibility_xgb = xgb_final.predict_proba(grid_X)[:, 1]
susceptibility_ensemble = (susceptibility_rf + susceptibility_xgb) / 2

tree_preds = np.stack(
    [t.predict_proba(grid_X_imputed)[:, 1] for t in rf_final.named_steps["clf"].estimators_], axis=1
)
rf_tree_uncertainty = tree_preds.std(axis=1)
model_disagreement = np.abs(susceptibility_rf - susceptibility_xgb)

grid_df["susceptibility"] = susceptibility_ensemble
grid_df["susceptibility_rf"] = susceptibility_rf
grid_df["susceptibility_xgb"] = susceptibility_xgb
grid_df["rf_tree_uncertainty"] = rf_tree_uncertainty
grid_df["model_disagreement"] = model_disagreement

output_cols = ["latitude", "longitude", "susceptibility", "susceptibility_rf", "susceptibility_xgb",
               "rf_tree_uncertainty", "model_disagreement"]
grid_df[output_cols].to_csv("phase5_susceptibility_grid.csv", index=False)
print(f"\nSaved phase5_susceptibility_grid.csv: {grid_df.shape[0]} points")
print("\nReminder: per the Phase 4 caveat and Section 2 SHAP findings, treat this map as showing")
print("'terrain-stability-weighted susceptibility' more than pure rainfall-triggering susceptibility")
print("until the Phase 2 negative-sampling redesign is done.")


## 5. Render the susceptibility + uncertainty rasters

Reshapes the grid CSV back into 2D arrays for plotting -- straightforward since `grid_points` was built
on a regular `GRID_STEP_DEG` lattice, so `pivot` recovers the raster shape directly (no interpolation
needed, unlike an irregular point set).

In [ ]:
import matplotlib.pyplot as plt

grid_full = pd.DataFrame({"latitude": [p[1] for p in grid_points], "longitude": [p[0] for p in grid_points]})
plot_df = grid_full.merge(grid_df[output_cols], on=["latitude", "longitude"], how="left")

susceptibility_raster = plot_df.pivot(index="latitude", columns="longitude", values="susceptibility")
uncertainty_raster = plot_df.pivot(index="latitude", columns="longitude", values="rf_tree_uncertainty")
disagreement_raster = plot_df.pivot(index="latitude", columns="longitude", values="model_disagreement")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
extent = [lon_min, lon_max, lat_min, lat_max]

im0 = axes[0].imshow(susceptibility_raster.values, origin="lower", extent=extent,
                      cmap="YlOrRd", vmin=0, vmax=1)
axes[0].set_title("Susceptibility (RF+XGB ensemble)")
plt.colorbar(im0, ax=axes[0], fraction=0.03)

im1 = axes[1].imshow(uncertainty_raster.values, origin="lower", extent=extent, cmap="Purples")
axes[1].set_title("Uncertainty: RF tree disagreement (std)")
plt.colorbar(im1, ax=axes[1], fraction=0.03)

im2 = axes[2].imshow(disagreement_raster.values, origin="lower", extent=extent, cmap="Blues")
axes[2].set_title("Uncertainty: RF/XGB disagreement")
plt.colorbar(im2, ax=axes[2], fraction=0.03)

for ax in axes:
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

plt.tight_layout()
plt.savefig("phase5_susceptibility_maps.png", dpi=150)
plt.show()
print("Saved phase5_susceptibility_maps.png")


**Reading these maps:** high susceptibility in a location where uncertainty is also high (either
panel) is the least trustworthy kind of result -- the model is guessing, not confidently identifying
risk. High susceptibility with low uncertainty on both measures is the most defensible signal. Given
the Section 2 finding, cross-reference any "high susceptibility" region against elevation before
trusting it -- if it lines up with mid-elevation valley terrain generally (not a specific rainfall/
vegetation signature), it may be reflecting the terrain-confound rather than genuine risk.